Code to set path root

In [ ]:
import sys
import os
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)


sys.path.append(os.path.abspath(".."))

# Training model on `fight-weaponized-other-dataset` with 64x64 Image Sizes
* ResNet Included
* using `datasets`, `transforms` module from `torchvison`
* using `dataloader` module from `torch.utils.data`

## Importing necessary Modules

In [ ]:
# Import torch libraries
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

# Import modules
from modules.architectures.Architecture import Architecture, ResidualBlock, Shortcut
from modules.helper.Trainer import Trainer
from modules.helper.Plotter import plot_training_metrics, plot_testing_history
from modules.helper.Tester import  Tester

Check if CUDA is used

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("CUDA device name:", torch.cuda.get_device_name(0))
    print("Current device index:", torch.cuda.current_device())
    print("Device count:", torch.cuda.device_count())
else:
    print("Running on CPU")

### Use datasets, dataloader and transforms for loading training Dataset

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/train",
    transform = train_transform
)

train_dataloader = DataLoader(
    dataset=train_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    shuffle=True
)

print("Total Batches => ", len(train_dataloader))

### Use datasets, dataloader and transforms for loading validation Dataset

In [ ]:
val_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

val_dataset = datasets.ImageFolder(
    root = "../datasets/fight-weaponized-other-dataset/val",
    transform = val_transform
)

val_dataloader = DataLoader(
    dataset=val_dataset,
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

print("Total Batches => ", len(val_dataloader))

### Using Model Architecture:
* 10 Convolutional Layers
    - Conv2D
    - BatchNorm2D
    - ReLu
    - MaxPool2D (Optional)
* 1 Linear Layer
* SDG Optimizer

In [ ]:
model = Architecture().to("cuda")

### Adding 100 blocks (MaxPool2D in each 20 block)

In [ ]:
out_channels = 8
size = 64

model_blocks = [
    nn.Conv2d(3, out_channels, 3, 1, 1),
    nn.BatchNorm2d(out_channels),
    nn.ReLU()
]

for stage in range(5):

    for i in range(10):

        conv = nn.Conv2d(out_channels, out_channels, 3, 1, 1)
        bn = nn.BatchNorm2d(out_channels)

        model_blocks.append(
            ResidualBlock([
                nn.Conv2d(out_channels, out_channels, 3, 1, 1),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                
                
                nn.Conv2d(out_channels, out_channels, 3, 1, 1),
                nn.BatchNorm2d(out_channels)
            ])
        )

    if stage < 3:
        model_blocks.append(nn.MaxPool2d(2, 2))
        size //= 2

    if stage < 4:
        new_channels = out_channels * 2

        model_blocks.append(
            ResidualBlock([
                nn.Conv2d(out_channels, new_channels, 3, 1, 1),
                nn.BatchNorm2d(new_channels),
                nn.ReLU(),
                ],
                Shortcut(out_channels, new_channels)
            )
        )

        out_channels = new_channels


print(f"Final Out Channels = {out_channels}")
print(f"Final Shape = {size}")

In [ ]:
model = model.add(
    # Conv Blocks
    *model_blocks,
    
    # Flatten
    nn.Flatten(),

    nn.Linear(out_channels * size * size, 128),
    nn.ReLU(),
    nn.Linear(128, 3)
    )

### Use Trainer to train and check validations
Adding weight decay and decreased weight

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=3e-4, weight_decay=1e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
trainer = Trainer(
    model, 
    train_dataloader, 
    val_dataloader, 
    optimizer=optimizer, 
    num_classes=3,
    criterion=criterion,
    device="cuda",
    save_dir="../models/experiment9/",
    save_checkpoints=10,
    print_every=5
    )

In [ ]:
history = trainer.fit(100)

### Save Metrics

In [ ]:
df = plot_training_metrics(history)
df.to_csv("../documentations/experiments/experiment9/tables/training_metrics.csv", index=False)

### Training/Validation Trend (100 epochs)
* The model learned rapidly during the initial phase, with training accuracy increasing from 48.82% at epoch 1 to 86.20% by epoch 10.
* Training loss decreased consistently from 1.009 at epoch 1 to 0.366 at epoch 10, showing strong optimization progress.
* Validation accuracy improved quickly in the early epochs, reaching 79.20% at epoch 18, indicating effective feature learning.
* Training and validation performance showed a noticeable gap after around epoch 20, with training accuracy continuing toward 100% while validation accuracy remained around 75–79%.
* The model achieved very high training performance, exceeding 99% accuracy after epoch 35 and reaching 100% accuracy multiple times in later epochs.
* Validation accuracy did not improve proportionally with training accuracy, indicating overfitting.
* Validation loss reached its lowest value at epoch 16 (0.6546), suggesting the best generalization occurred earlier than the final epochs.
* The model's best validation accuracy was 79.42% at epoch 48 and epoch 73.
* Validation F1 score peaked at epoch 73 with approximately 79.58%, showing the strongest balanced classification performance.
* After approximately epoch 50, validation loss became unstable and frequently increased above 0.9, despite near-perfect training metrics.
* The model continued memorizing training data after reaching convergence, with training loss dropping below 0.01 in later epochs while validation performance fluctuated.
* Large validation loss spikes occurred around epochs 54, 66, 67, 79, and 83, showing unstable generalization.
* Confusion matrices remained relatively balanced across classes, but some class confusion persisted throughout training.
* The third class showed occasional higher misclassification rates, especially during unstable validation periods.
* Precision, recall, and F1 score followed the same pattern as accuracy: strong early improvement followed by saturation.
* The model reached its most reliable generalization region between epochs 16 and 73 rather than at the final epoch.
* Continuing training beyond the point of validation peak reduced generalization quality while improving only training metrics.

The model showed strong learning capability, with rapid improvement during early training and steady reduction in training loss. It learned useful features quickly, reaching over 90% training accuracy by epoch 14 and nearly perfect training performance later. However, after approximately epoch 20, the model began overfitting as training metrics continued improving while validation metrics remained mostly stable. The best generalization occurred around epoch 73, where the model achieved the highest validation F1 score and strong validation accuracy. Further training increased memorization without improving real-world performance, suggesting that early stopping around the validation peak would provide a better final model.

<b>Best Epoch 73</b>

<b>Loss</b>
* Train Loss = 0.010490482062962662
* Valid Loss = 0.869190007099954

<b>Training Metrics</b>
* Train Accuracy = 0.9990548491477966
* Train Precison = 0.9990338683128357
* Train Recall = 0.9990338683128357
* Train F1 = 0.9990338683128357

<b>Validation Accuracy</b>
* Validation Accuracy = 0.7942478060722351
* Validation Precision = 0.7978400588035583
* Validation Recall = 0.7944918274879456
* Validation F1 = 0.7957541942596436

## Use Tester Module to Test Model

Load Model with State Dict

In [ ]:
# Transforms of Data
test_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# Dataset Loading From Image dir
test_dataset = datasets.ImageFolder(
    root="../datasets/fight-weaponized-other-dataset/test", 
    transform = test_transform 
    )

# DataLoader
test_loader = DataLoader(
    dataset=test_dataset, 
    batch_size=16,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
    )

tester = Tester(
    model,
    test_loader,
    3,
    torch.nn.CrossEntropyLoss(),
    "cuda"
)

test_scores = tester.test_all_checkpoints(
    "../models/experiment9"
)

### Save Test Metrics

In [ ]:
# Plot all 100 epochs
test_metrics_df = plot_testing_history(test_scores)
test_metrics_df.to_csv("../documentations/experiments/experiment9/tables/test_metrics.csv", index=False)

### Test Performance Trend
* The model started with a strong baseline performance, achieving 58.55% test accuracy and 58.44% test F1 score at epoch 1.
* The model learned quickly during the initial training phase, improving test accuracy to 73.25% by epoch 5.
* Test loss decreased rapidly from 0.924 at epoch 1 to around 0.585 by epoch 14, indicating improved classification confidence.
* Test accuracy exceeded 78% by epoch 7 and remained consistently near 80% for the rest of training.
* The model achieved its first strong performance point at epoch 14 with 79.82% accuracy and 80.02% F1 score.
* After epoch 14, the model entered a stable convergence stage where performance fluctuations remained small.
* The model showed a temporary performance decline around epoch 30, where accuracy dropped to 76.10%, but recovered afterward.
* From epoch 40 onward, the model maintained stable test accuracy between approximately 79% and 81%.
* Test precision remained consistently higher than or close to accuracy, indicating reliable class predictions.
* Test recall closely matched precision throughout training, showing balanced classification behaviour.
* The model showed no major prediction collapse or extreme instability during training.
* The highest test accuracy was achieved at epochs 50 and 60 with 80.48%.
* The highest test F1 score was achieved at epoch 60 with 80.75%, making it the strongest overall generalization point.
* Performance after epoch 60 remained stable but did not improve, suggesting that the model had reached convergence.
* Epoch 100 showed slightly reduced performance compared with the best epoch, indicating that additional training did not provide
* further generalization improvement.Overall, the model demonstrated efficient learning, stable convergence, and good resistance to overfitting.

The model showed strong and stable test performance throughout training. It improved rapidly from 58.55% accuracy at the beginning to over 80% accuracy within the first 60 epochs. Unlike models with large validation or test fluctuations, this model maintained consistent accuracy, precision, recall, and F1 values after convergence. The best generalization performance was achieved at epoch 60, where the model reached the highest test F1 score of 0.8075 along with balanced precision and recall. Training beyond this point did not improve performance and slightly reduced the final test metrics.

<b>Best Epoch 60</b>

* Loss = 0.7126405369653467
* Accuracy = 0.8048245906829834
* Precision = 0.8111618161201472
* Recall = 0.8053563237190247
* F1-Score = 0.8074808120727539